# EXP-001: BSD10k Training Data Confidence Filtering (>= 4)

**가설**: BSD10k 훈련 데이터에서 confidence >= 4 샘플만 사용하면 노이즈가 줄어 H-Acc가 상승한다.

**근거**: 논문 A (HATR, Anastasopoulou et al.) — confidence >= 4 필터링 시 2nd level +9%, top level +5%

**비교 기준 (EXP-000 baseline)**:
- H-Acc: 79.45%
- Macro 2nd: 74.02%
- Top Acc: 88.88%

**설계**:
- 5-fold split은 BSD10k 전체(confidence 무관)로 동일하게 유지 → test set = baseline과 동일
- 각 fold의 **훈련 데이터만** confidence >= 4로 필터링
- test는 confidence 무관 (전체 BSD10k test fold 사용) → 공정한 비교

In [2]:
# ============================================================
# CONFIGURATION — 여기만 수정하면 됩니다
# ============================================================

SEED = 1821                    # baseline과 동일한 seed (fold split 재현)
CONFIDENCE_THRESHOLD = 4       # 훈련에 사용할 최소 confidence
MODE = 'both'                  # 'both' | 'audio' | 'text'
K_FOLDS = 5
NUM_EPOCHS = 100
BATCH_SIZE = 64
LR = 0.001
HIDDEN_SIZE = 128
DROPOUT = 0.1
SCHEDULER_TYPE = 'step'
PATIENCE = 5
EARLY_STOPPING_FACTOR = 3

EXP_ID = 'exp_001'
EXP_DESC = f'BSD10k confidence>={CONFIDENCE_THRESHOLD} 훈련 필터링 (both mode, 5-fold)'
MODEL_OUTPUT_DIR = '../model_output_exp001'

In [3]:
import sys, os

# 코드 모듈 경로 추가 (어디서 실행해도 동작)
NOTEBOOK_DIR = os.path.abspath('')
CODE_DIR = os.path.join(NOTEBOOK_DIR, 'dcase2026_task1_baseline')
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import json
import random
from collections import defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from torch.utils.data import DataLoader

# baseline 코드 임포트
from models import BaseClassifier
from dataset_utils import HATRDataset
from evaluate import evaluate_model
from losses import CrossEntropyLoss
from utils import (
    set_seed,
    build_class_to_topclass_mapping,
    build_class_to_topclass_tensor,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
CUDA available: True
GPU: NVIDIA GeForce RTX 3060


In [4]:
# ============================================================
# 경로 설정
# ============================================================

PROJECT_ROOT = NOTEBOOK_DIR

PROCESSED_DATASET_CSV  = os.path.join(PROJECT_ROOT, 'dcase2026_task1_baseline', 'data', 'processed_dataset.csv')
BSD10K_METADATA_CSV    = os.path.join(PROJECT_ROOT, 'data', 'metadata', 'BSD10k_metadata.csv')
CLASS_DICT_JSON        = os.path.join(PROJECT_ROOT, 'dcase2026_task1_baseline', 'data', 'class_dict.json')
TOP_CLASS_DICT_JSON    = os.path.join(PROJECT_ROOT, 'dcase2026_task1_baseline', 'data', 'top_class_dict.json')
EXPERIMENTS_DIR        = os.path.join(PROJECT_ROOT, 'experiments')

os.makedirs(EXPERIMENTS_DIR, exist_ok=True)
os.makedirs(MODEL_OUTPUT_DIR if os.path.isabs(MODEL_OUTPUT_DIR) else
            os.path.join(PROJECT_ROOT, MODEL_OUTPUT_DIR.lstrip('../')), exist_ok=True)

for path, name in [
    (PROCESSED_DATASET_CSV, 'processed_dataset.csv'),
    (BSD10K_METADATA_CSV,   'BSD10k_metadata.csv'),
    (CLASS_DICT_JSON,       'class_dict.json'),
    (TOP_CLASS_DICT_JSON,   'top_class_dict.json'),
]:
    exists = '✓' if os.path.exists(path) else '✗ NOT FOUND'
    print(f'  [{exists}] {name}')

with open(CLASS_DICT_JSON)     as f: class_dict     = json.load(f)
with open(TOP_CLASS_DICT_JSON) as f: top_class_dict = json.load(f)

print(f'\nClasses: {len(class_dict)} (2nd level), {len(top_class_dict)} (top level)')

  [✓] processed_dataset.csv
  [✓] BSD10k_metadata.csv
  [✓] class_dict.json
  [✓] top_class_dict.json

Classes: 23 (2nd level), 5 (top level)


In [5]:
# ============================================================
# BSD10k 데이터 로드 + confidence 정보 병합
# ============================================================

full_df   = pd.read_csv(PROCESSED_DATASET_CSV)
meta_df   = pd.read_csv(BSD10K_METADATA_CSV)
meta_df['sound_id'] = meta_df['sound_id'].astype(str).str.strip()
full_df['index']    = full_df['index'].astype(str).str.strip()

# confidence 컬럼 병합
full_df = full_df.merge(
    meta_df[['sound_id', 'confidence']],
    left_on='index', right_on='sound_id',
    how='left'
).drop(columns=['sound_id'])

print(f'전체 BSD10k 샘플: {len(full_df)}')
print(f'\nConfidence 분포:')
print(full_df['confidence'].value_counts().sort_index())

conf_ge4 = full_df[full_df['confidence'] >= CONFIDENCE_THRESHOLD]
print(f'\nconfidence >= {CONFIDENCE_THRESHOLD}: {len(conf_ge4)}개 ({100*len(conf_ge4)/len(full_df):.1f}%)')
print(f'필터링 후 제외 샘플: {len(full_df) - len(conf_ge4)}개')

전체 BSD10k 샘플: 10956

Confidence 분포:
confidence
1     106
2     749
3    3280
4    6045
5     776
Name: count, dtype: int64

confidence >= 4: 6821개 (62.3%)
필터링 후 제외 샘플: 4135개


In [6]:
# 클래스별 confidence >= 4 비율 확인
class_conf = full_df.groupby('class').apply(
    lambda g: (g['confidence'] >= CONFIDENCE_THRESHOLD).mean()
).sort_values()

print('클래스별 confidence >= 4 비율 (낮은 순):')
for cls, ratio in class_conf.items():
    print(f'  {cls}: {ratio:.1%}')

print(f'\n평균: {class_conf.mean():.1%}, 최소: {class_conf.min():.1%}')

클래스별 confidence >= 4 비율 (낮은 순):
  fx-ex: 30.5%
  ss-i: 34.8%
  ss-u: 39.4%
  sp-c: 41.1%
  sp-p: 41.8%
  is-e: 45.8%
  ss-n: 53.7%
  ss-s: 54.4%
  fx-a: 55.0%
  m-si: 58.2%
  fx-m: 59.1%
  m-m: 60.3%
  fx-v: 60.8%
  fx-h: 64.1%
  fx-el: 65.9%
  fx-o: 66.0%
  fx-n: 68.8%
  m-sp: 69.5%
  is-k: 71.0%
  sp-s: 72.6%
  is-p: 77.0%
  is-w: 77.9%
  is-s: 86.7%

평균: 58.9%, 최소: 30.5%


In [7]:
# ============================================================
# 헬퍼 함수 (train_test.py와 동일)
# ============================================================

def init_weights(model):
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)


def make_serializable(obj, decimals=6):
    if isinstance(obj, torch.Tensor):
        return make_serializable(obj.detach().cpu().numpy(), decimals)
    elif isinstance(obj, np.ndarray):
        return round(float(obj), decimals) if obj.ndim == 0 else [make_serializable(x, decimals) for x in obj]
    elif isinstance(obj, float):
        return round(obj, decimals)
    elif isinstance(obj, int):
        return obj
    elif isinstance(obj, dict):
        return {k: make_serializable(v, decimals) for k, v in obj.items()}
    elif hasattr(obj, '__iter__') and not isinstance(obj, (str, bytes)):
        return [make_serializable(x, decimals) for x in obj]
    return obj


def train_model(model, train_loader, val_loader, device,
                num_epochs=100, lr=0.001, criterion=None,
                output_dir='model_output', scheduler_type='step',
                patience=5, early_stopping_factor=3):
    os.makedirs(output_dir, exist_ok=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

    if scheduler_type == 'plateau':
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=0.5, patience=patience, verbose=False)
    elif scheduler_type == 'step':
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
    else:
        scheduler = None

    best_accuracy = 0.0
    no_improve = 0
    history = defaultdict(list)

    for epoch in range(num_epochs):
        model.train()
        total_loss, total_samples = 0.0, 0

        for data in train_loader:
            labels    = data['class_idx'].to(device)
            audio_emb = data.get('audio_embedding')
            text_emb  = data.get('text_embedding')
            if audio_emb is not None: audio_emb = audio_emb.to(device)
            if text_emb  is not None: text_emb  = text_emb.to(device)

            optimizer.zero_grad()
            _, logits, _ = model(audio_emb, text_emb)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            bs = labels.size(0)
            total_loss    += loss.item() * bs
            total_samples += bs

        history['train_loss'].append(total_loss / total_samples)
        history['lr'].append(optimizer.param_groups[0]['lr'])

        # validation
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for data in val_loader:
                labels    = data['class_idx'].to(device)
                audio_emb = data.get('audio_embedding')
                text_emb  = data.get('text_embedding')
                if audio_emb is not None: audio_emb = audio_emb.to(device)
                if text_emb  is not None: text_emb  = text_emb.to(device)
                _, logits, _ = model(audio_emb, text_emb)
                preds = torch.argmax(logits, dim=1)
                correct += (preds == labels).sum().item()
                total   += labels.size(0)

        val_acc = 100 * correct / total
        history['val_accuracy'].append(val_acc)
        print(f'  Epoch [{epoch+1:3d}/{num_epochs}] loss={total_loss/total_samples:.4f}  val_acc={val_acc:.2f}%')

        if scheduler:
            if scheduler_type == 'plateau': scheduler.step(val_acc)
            else: scheduler.step()

        if val_acc > best_accuracy:
            best_accuracy = val_acc
            no_improve = 0
            cfg = {
                'hidden_size': model.hidden_size,
                'num_classes': model.num_classes,
                'emb_size_audio': model.emb_size_audio,
                'emb_size_text': model.emb_size_text,
                'dropout': model.dropout,
                'use_batch_norm': True,
                'mode': model.mode,
            }
            torch.save({'model_state': model.state_dict(), 'config': cfg},
                       os.path.join(output_dir, 'best_model.pth'))
            print(f'    → best model saved ({val_acc:.2f}%)')
        else:
            no_improve += 1
            if no_improve >= patience * early_stopping_factor:
                print('  Early stopping.')
                break

    with open(os.path.join(output_dir, 'history.json'), 'w') as f:
        json.dump(make_serializable(history), f, indent=2)

    return best_accuracy, history, model

print('Helper functions defined.')

Helper functions defined.


In [9]:
# ============================================================
# 5-FOLD 학습 메인 루프
# ============================================================

seed = set_seed(SEED)
class_to_top = build_class_to_topclass_mapping(class_dict, top_class_dict)

labels = full_df['class_idx'].tolist()
skf    = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=seed)

fold_results = []

for fold, (trainval_idx, test_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
    print(f"\n{'='*60}")
    print(f"FOLD {fold} / {K_FOLDS-1}")
    print(f"{'='*60}")


    # trainval → train / val split (80/20)
    trainval_labels = [labels[i] for i in trainval_idx]
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    train_rel, val_rel = next(sss.split(np.zeros(len(trainval_labels)), trainval_labels))
    train_idx_abs = [trainval_idx[i] for i in train_rel]
    val_idx_abs   = [trainval_idx[i] for i in val_rel]

    # ── confidence 필터링: train 데이터만 적용 ──
    train_df_full = full_df.iloc[train_idx_abs].reset_index(drop=True)
    train_df = train_df_full[train_df_full['confidence'] >= CONFIDENCE_THRESHOLD].reset_index(drop=True)

    val_df  = full_df.iloc[val_idx_abs].reset_index(drop=True)
    test_df = full_df.iloc[test_idx].reset_index(drop=True)

    print(f'Train (before filter): {len(train_df_full)}  →  after conf>={CONFIDENCE_THRESHOLD}: {len(train_df)} ({100*len(train_df)/len(train_df_full):.1f}%)')
    print(f'Val:  {len(val_df)}   Test: {len(test_df)}')

    # ── Dataset & DataLoader ──
    train_dataset = HATRDataset(train_df, aug=True,  mask_pct=0.7)
    val_dataset   = HATRDataset(val_df,   aug=False)
    test_dataset  = HATRDataset(test_df,  aug=False)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              drop_last=True, num_workers=0, pin_memory=torch.cuda.is_available())
    val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=0, pin_memory=torch.cuda.is_available())
    test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=0, pin_memory=torch.cuda.is_available())

    # ── 모델 ──
    emb_audio = 512 if MODE in ['audio', 'both'] else 0
    emb_text  = 512 if MODE in ['text',  'both'] else 0

    model = BaseClassifier(
        hidden_size=HIDDEN_SIZE,
        num_classes=len(class_dict),
        emb_size_audio=emb_audio,
        emb_size_text=emb_text,
        dropout=DROPOUT,
        use_batch_norm=True,
        mode=MODE
    ).to(device)
    init_weights(model)

    criterion  = CrossEntropyLoss()
    output_dir = os.path.join(
        MODEL_OUTPUT_DIR if os.path.isabs(MODEL_OUTPUT_DIR) else
        os.path.join(PROJECT_ROOT, MODEL_OUTPUT_DIR.lstrip('../')),
        MODE, f'fold_{fold}'
    )

    # ── 학습 ──
    best_acc, history, trained_model = train_model(
        model, train_loader, val_loader, device,
        num_epochs=NUM_EPOCHS, lr=LR,
        criterion=criterion,
        output_dir=output_dir,
        scheduler_type=SCHEDULER_TYPE,
        patience=PATIENCE,
        early_stopping_factor=EARLY_STOPPING_FACTOR
    )

    # ── 평가 ──
    model_path = os.path.join(output_dir, 'best_model.pth')
    metrics = evaluate_model(
        BaseClassifier, model_path, test_loader, device,
        class_to_top, output_dir=output_dir, fold_id=fold, class_dict=class_dict
    )

    fold_results.append({
        'fold': fold,
        'train_size': len(train_df),
        'val_size':   len(val_df),
        'test_size':  len(test_df),
        'accuracy':            metrics['accuracy'],
        'top_accuracy':        metrics['top_accuracy'],
        'macro_accuracy':      metrics['macro_accuracy'],
        'macro_top_accuracy':  metrics['macro_top_accuracy'],
        'hierarchical_acc':    metrics['hierarchical_accuracy'],
        'hierarchical_f1':     metrics['hierarchical_f1'],
    })

    print(f'\n── Fold {fold} Results ──')
    print(f'  Accuracy:        {metrics["accuracy"]:.2f}%')
    print(f'  Top Accuracy:    {metrics["top_accuracy"]:.2f}%')
    print(f'  Macro 2nd:       {metrics["macro_accuracy"]:.2f}%')
    print(f'  H-Acc:           {metrics["hierarchical_accuracy"]:.2f}%')
    print(f'  H-F1:            {metrics["hierarchical_f1"]:.2f}%')

print('\n모든 fold 완료!')


FOLD 0 / 4
Train (before filter): 7011  →  after conf>=4: 4425 (63.1%)
Val:  1753   Test: 2192
  Epoch [  1/100] loss=2.6751  val_acc=43.30%
    → best model saved (43.30%)
  Epoch [  2/100] loss=1.6146  val_acc=61.44%
    → best model saved (61.44%)
  Epoch [  3/100] loss=1.2488  val_acc=68.85%
    → best model saved (68.85%)
  Epoch [  4/100] loss=1.0401  val_acc=70.39%
    → best model saved (70.39%)
  Epoch [  5/100] loss=0.8881  val_acc=71.82%
    → best model saved (71.82%)
  Epoch [  6/100] loss=0.7960  val_acc=72.85%
    → best model saved (72.85%)
  Epoch [  7/100] loss=0.7479  val_acc=71.93%
  Epoch [  8/100] loss=0.7138  val_acc=73.59%
    → best model saved (73.59%)
  Epoch [  9/100] loss=0.6499  val_acc=73.76%
    → best model saved (73.76%)
  Epoch [ 10/100] loss=0.6072  val_acc=73.19%
  Epoch [ 11/100] loss=0.5675  val_acc=74.84%
    → best model saved (74.84%)
  Epoch [ 12/100] loss=0.5377  val_acc=75.30%
    → best model saved (75.30%)
  Epoch [ 13/100] loss=0.5534  v

c:\Users\solok\Desktop\Dcase baseline\dcase2026_task1_baseline\evaluate.py:97: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location

[BaseClassifier | Fold 0] Accuracy: 77.69%
[BaseClassifier | Fold 0] Top class accuracy: 88.37%
[BaseClassifier | Fold 0] Macro accuracy: 71.11%
[BaseClassifier | Fold 0] Macro top class accuracy: 84.52%
[BaseClassifier | Fold 0] Hierarchical accuracy: 77.82%
[BaseClassifier | Fold 0] Hierarchical precision: 77.75%
[BaseClassifier | Fold 0] Hierarchical recall: 76.14%
[BaseClassifier | Fold 0] Hierarchical F1: 76.68%

── Fold 0 Results ──
  Accuracy:        77.69%
  Top Accuracy:    88.37%
  Macro 2nd:       71.11%
  H-Acc:           77.82%
  H-F1:            76.68%

FOLD 1 / 4
Train (before filter): 7012  →  after conf>=4: 4329 (61.7%)
Val:  1753   Test: 2191
  Epoch [  1/100] loss=2.6087  val_acc=46.66%
    → best model saved (46.66%)
  Epoch [  2/100] loss=1.6139  val_acc=61.04%
    → best model saved (61.04%)
  Epoch [  3/100] loss=1.2113  val_acc=66.91%
    → best model saved (66.91%)
  Epoch [  4/100] loss=0.9428  val_acc=68.34%
    → best model saved (68.34%)
  Epoch [  5/100] l

c:\Users\solok\Desktop\Dcase baseline\dcase2026_task1_baseline\evaluate.py:97: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location

[BaseClassifier | Fold 1] Accuracy: 78.87%
[BaseClassifier | Fold 1] Top class accuracy: 88.64%
[BaseClassifier | Fold 1] Macro accuracy: 72.64%
[BaseClassifier | Fold 1] Macro top class accuracy: 85.06%
[BaseClassifier | Fold 1] Hierarchical accuracy: 78.85%
[BaseClassifier | Fold 1] Hierarchical precision: 77.20%
[BaseClassifier | Fold 1] Hierarchical recall: 77.30%
[BaseClassifier | Fold 1] Hierarchical F1: 77.00%

── Fold 1 Results ──
  Accuracy:        78.87%
  Top Accuracy:    88.64%
  Macro 2nd:       72.64%
  H-Acc:           78.85%
  H-F1:            77.00%

FOLD 2 / 4
Train (before filter): 7012  →  after conf>=4: 4372 (62.4%)
Val:  1753   Test: 2191
  Epoch [  1/100] loss=2.5428  val_acc=52.71%
    → best model saved (52.71%)
  Epoch [  2/100] loss=1.4860  val_acc=65.15%
    → best model saved (65.15%)
  Epoch [  3/100] loss=1.1334  val_acc=72.50%
    → best model saved (72.50%)
  Epoch [  4/100] loss=0.9743  val_acc=73.59%
    → best model saved (73.59%)
  Epoch [  5/100] l

...

[BaseClassifier | Fold 2] Accuracy: 79.42%
[BaseClassifier | Fold 2] Top class accuracy: 89.23%
[BaseClassifier | Fold 2] Macro accuracy: 73.56%
[BaseClassifier | Fold 2] Macro top class accuracy: 85.56%
[BaseClassifier | Fold 2] Hierarchical accuracy: 79.56%
[BaseClassifier | Fold 2] Hierarchical precision: 78.90%
[BaseClassifier | Fold 2] Hierarchical recall: 78.06%
[BaseClassifier | Fold 2] Hierarchical F1: 78.31%

── Fold 2 Results ──
  Accuracy:        79.42%
  Top Accuracy:    89.23%
  Macro 2nd:       73.56%
  H-Acc:           79.56%
  H-F1:            78.31%

FOLD 3 / 4
Train (before filter): 7012  →  after conf>=4: 4358 (62.2%)
Val:  1753   Test: 2191
  Epoch [  1/100] loss=2.5594  val_acc=50.43%
    → best model saved (50.43%)
  Epoch [  2/100] loss=1.5389  val_acc=60.98%
    → best model saved (60.98%)
  Epoch [  3/100] loss=1.2176  val_acc=69.88%
    → best model saved (69.88%)
  Epoch [  4/100] loss=1.0170  val_acc=70.39%
    → best model saved (70.39%)
  Epoch [  5/100] l

c:\Users\solok\Desktop\Dcase baseline\dcase2026_task1_baseline\evaluate.py:97: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location

[BaseClassifier | Fold 3] Accuracy: 78.32%
[BaseClassifier | Fold 3] Top class accuracy: 88.09%
[BaseClassifier | Fold 3] Macro accuracy: 72.17%
[BaseClassifier | Fold 3] Macro top class accuracy: 85.13%
[BaseClassifier | Fold 3] Hierarchical accuracy: 78.65%
[BaseClassifier | Fold 3] Hierarchical precision: 78.37%
[BaseClassifier | Fold 3] Hierarchical recall: 77.03%
[BaseClassifier | Fold 3] Hierarchical F1: 76.94%

── Fold 3 Results ──
  Accuracy:        78.32%
  Top Accuracy:    88.09%
  Macro 2nd:       72.17%
  H-Acc:           78.65%
  H-F1:            76.94%

FOLD 4 / 4
Train (before filter): 7012  →  after conf>=4: 4351 (62.1%)
Val:  1753   Test: 2191
  Epoch [  1/100] loss=2.6424  val_acc=46.95%
    → best model saved (46.95%)
  Epoch [  2/100] loss=1.4903  val_acc=63.43%
    → best model saved (63.43%)
  Epoch [  3/100] loss=1.0977  val_acc=69.31%
    → best model saved (69.31%)
  Epoch [  4/100] loss=0.9014  val_acc=70.56%
    → best model saved (70.56%)
  Epoch [  5/100] l

c:\Users\solok\Desktop\Dcase baseline\dcase2026_task1_baseline\evaluate.py:97: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location

[BaseClassifier | Fold 4] Accuracy: 78.05%
[BaseClassifier | Fold 4] Top class accuracy: 86.90%
[BaseClassifier | Fold 4] Macro accuracy: 72.05%
[BaseClassifier | Fold 4] Macro top class accuracy: 82.68%
[BaseClassifier | Fold 4] Hierarchical accuracy: 77.37%
[BaseClassifier | Fold 4] Hierarchical precision: 76.37%
[BaseClassifier | Fold 4] Hierarchical recall: 76.04%
[BaseClassifier | Fold 4] Hierarchical F1: 75.93%

── Fold 4 Results ──
  Accuracy:        78.05%
  Top Accuracy:    86.90%
  Macro 2nd:       72.05%
  H-Acc:           77.37%
  H-F1:            75.93%

모든 fold 완료!


In [10]:
# ============================================================
# 결과 집계 및 baseline 비교
# ============================================================

BASELINE = {
    'accuracy':       79.63,
    'top_accuracy':   88.88,
    'macro_accuracy': 74.02,
    'hierarchical_acc': 79.45,
    'hierarchical_f1':  78.33,
}

keys = ['accuracy', 'top_accuracy', 'macro_accuracy', 'hierarchical_acc', 'hierarchical_f1']
avg = {k: np.mean([r[k] for r in fold_results]) for k in keys}

print('\n' + '='*60)
print('EXP-001 결과 요약')
print('='*60)
print(f"{'Metric':<22} {'Baseline':>10} {'EXP-001':>10} {'Diff':>8}")
print('-'*60)
labels_map = {
    'accuracy':         'Accuracy (micro)',
    'top_accuracy':     'Top Accuracy',
    'macro_accuracy':   'Macro 2nd',
    'hierarchical_acc': 'H-Acc ★',
    'hierarchical_f1':  'H-F1',
}
for k in keys:
    diff = avg[k] - BASELINE[k]
    arrow = '↑' if diff > 0 else ('↓' if diff < 0 else '=')
    print(f"{labels_map[k]:<22} {BASELINE[k]:>9.2f}% {avg[k]:>9.2f}% {arrow}{abs(diff):>6.2f}%")

print('='*60)
print(f'\nFold별 H-Acc:')
for r in fold_results:
    print(f"  Fold {r['fold']}: {r['hierarchical_acc']:.2f}%")
print(f"  평균:  {avg['hierarchical_acc']:.2f}%")


EXP-001 결과 요약
Metric                   Baseline    EXP-001     Diff
------------------------------------------------------------
Accuracy (micro)           79.63%     78.47% ↓  1.16%
Top Accuracy               88.88%     88.24% ↓  0.64%
Macro 2nd                  74.02%     72.31% ↓  1.71%
H-Acc ★                    79.45%     78.45% ↓  1.00%
H-F1                       78.33%     76.97% ↓  1.36%

Fold별 H-Acc:
  Fold 0: 77.82%
  Fold 1: 78.85%
  Fold 2: 79.56%
  Fold 3: 78.65%
  Fold 4: 77.37%
  평균:  78.45%


In [11]:
# ============================================================
# 실험 결과 JSON 저장
# ============================================================

import subprocess
try:
    git_commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'],
                                         cwd=PROJECT_ROOT).decode().strip()
except:
    git_commit = 'unknown'

h_acc_avg    = avg['hierarchical_acc']
h_acc_base   = BASELINE['hierarchical_acc']
vs_baseline  = f'{h_acc_avg - h_acc_base:+.2f}%'

result_json = {
    'exp_id':      EXP_ID,
    'branch':      'exp/confidence-filter',
    'description': EXP_DESC,
    'config': {
        'confidence_threshold': CONFIDENCE_THRESHOLD,
        'seed':         SEED,
        'hidden_size':  HIDDEN_SIZE,
        'mode':         MODE,
        'epochs':       NUM_EPOCHS,
        'batch_size':   BATCH_SIZE,
        'lr':           LR,
        'dropout':      DROPOUT,
        'scheduler':    SCHEDULER_TYPE,
    },
    'results': {
        'fold_avg': {
            'macro_acc_2nd':      round(avg['macro_accuracy'], 4),
            'macro_acc_top':      round(avg['top_accuracy'], 4),
            'hierarchical_acc':   round(avg['hierarchical_acc'], 4),
            'hierarchical_f1':    round(avg['hierarchical_f1'], 4),
        },
        'fold_details': [
            {
                'fold':          r['fold'],
                'train_size':    r['train_size'],
                'test_size':     r['test_size'],
                'accuracy':      round(r['accuracy'], 4),
                'top_accuracy':  round(r['top_accuracy'], 4),
                'macro_acc':     round(r['macro_accuracy'], 4),
                'h_acc':         round(r['hierarchical_acc'], 4),
                'h_f1':          round(r['hierarchical_f1'], 4),
            }
            for r in fold_results
        ]
    },
    'vs_baseline_h_acc': vs_baseline,
    'baseline_h_acc':    h_acc_base,
    'timestamp':         datetime.now().strftime('%Y-%m-%d %H:%M'),
    'git_commit':        git_commit,
}

out_path = os.path.join(EXPERIMENTS_DIR, f'{EXP_ID}_conf_filter.json')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(result_json, f, indent=2, ensure_ascii=False)

print(f'결과 저장 완료: {out_path}')
print(f'\n최종 H-Acc: {h_acc_avg:.2f}%  (baseline: {h_acc_base:.2f}%  →  {vs_baseline})')

결과 저장 완료: c:\Users\solok\Desktop\Dcase baseline\experiments\exp_001_conf_filter.json

최종 H-Acc: 78.45%  (baseline: 79.45%  →  -1.00%)
